<table style="width: 100%;">
    <tr style="background-color: transparent;"><td>
        <img src="https://data-88e.github.io/assets/images/blue_text.png" width="250px" style="margin-left: 0;" />
    </td><td>
        <p style="text-align: right; font-size: 10pt;"><strong>Economic Models</strong>, Fall 2023<br>
            Dr. Eric Van Dusen <br>
        ></td></tr>
</table>

## Capital Assets Pricing Model

In [56]:
try:
    import yfinance as yf
except:
    !pip install yfinance
    import yfinance as yf

In [57]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
import plotly.express as px

- Define your stock and market index symbols, as well as the risk-free rate (usually the 10-year Treasury yield).

- Retrieve historical data for your chosen stock and the market index using a library like pandas_datareader or by loading a CSV file with historical price data.

- Calculate the daily returns for both the stock and the market index.

- Calculate the excess returns of the stock by subtracting the risk-free rate from the stock's daily returns.

- Calculate the excess returns of the market by subtracting the risk-free rate from the market index's daily returns.

- Calculate the beta coefficient, which measures the stock's volatility relative to the market index. You can use linear regression to estimate the beta.

- Calculate the expected return of the stock using the CAPM formula:

In [58]:
# Define stock and market index symbols
stock_symbol = 'AAPL'
market_index_symbol = '^GSPC'  # S&P 500 index
risk_free_rate = 0.02  # 10-year Treasury yield

# Define the date range
start_date = '2020-01-01'
end_date = '2025-10-31'

In [59]:

# Download historical data using yfinance
stock_data = yf.download(stock_symbol, start=start_date, end=end_date)
market_data = yf.download(market_index_symbol, start=start_date, end=end_date)


/var/folders/wx/mgl11c114vv7vpz1f_b7szwr0000gn/T/ipykernel_37133/1856998602.py:2: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed
/var/folders/wx/mgl11c114vv7vpz1f_b7szwr0000gn/T/ipykernel_37133/1856998602.py:3: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed

/var/folders/wx/mgl11c114vv7vpz1f_b7szwr0000gn/T/ipykernel_37133/1856998602.py:3: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed


In [60]:
stock_data


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2020-01-02,72.468262,72.528582,71.223259,71.476600,135480400
2020-01-03,71.763718,72.523746,71.539330,71.696160,146322800
2020-01-06,72.335541,72.374146,70.634524,70.885457,118387200
2020-01-07,71.995361,72.600968,71.775796,72.345212,108872000
2020-01-08,73.153503,73.455103,71.698589,71.698589,132079200
...,...,...,...,...,...
2025-10-24,262.565491,263.874220,258.929001,260.937064,38253700
2025-10-27,268.549652,268.859349,264.393677,264.623466,44888200


In [61]:
stock_df = stock_data.reset_index()
market_df = market_data.reset_index()

# Flatten column names if they are MultiIndex
if isinstance(stock_df.columns, pd.MultiIndex):
    stock_df.columns = ['Date' if col[0] == 'Date' else col[0] for col in stock_df.columns]
if isinstance(market_df.columns, pd.MultiIndex):
    market_df.columns = ['Date' if col[0] == 'Date' else col[0] for col in market_df.columns]

# Create line plots for stock closing price and market index closing price
fig_stock = px.line(stock_df, x='Date', y='Close', title=f'{stock_symbol} Closing Price')
fig_stock.update_xaxes(title_text='Date')
fig_stock.update_yaxes(title_text='Price (USD)')

fig_market = px.line(market_df, x='Date', y='Close', title=f'{market_index_symbol} Closing Price')
fig_market.update_xaxes(title_text='Date')
fig_market.update_yaxes(title_text='Price (USD)')

# Show the plots
fig_stock.show()
fig_market.show()


In [62]:
# Calculate daily returns
# Based on the data structure shown, columns are like ('Close', 'AAPL'), ('High', 'AAPL'), etc.
# We'll use Close price for returns calculation since Adj Close may not be present
stock_returns = stock_data[('Close', stock_symbol)].pct_change().dropna()
market_returns = market_data[('Close', market_index_symbol)].pct_change().dropna()
market_returns


Date
2020-01-03   -0.007060
2020-01-06    0.003533
2020-01-07   -0.002803
2020-01-08    0.004902
2020-01-09    0.006655
                ...   
2025-10-24    0.007902
2025-10-27    0.012290
2025-10-28    0.002288
2025-10-29   -0.000044
2025-10-30   -0.009905
Name: (Close, ^GSPC), Length: 1465, dtype: float64

In [63]:
SR = pd.DataFrame({'Date': stock_returns.index, 'Returns': stock_returns})

# Create a line plot using Plotly Express
fig = px.line(SR, x='Date', y='Returns', title=f'{stock_symbol} Daily Returns')
fig.update_xaxes(title_text='Date')
fig.update_yaxes(title_text='Returns')

# Show the plot
fig.show()

In [64]:
# Create histogram of daily returns
fig_hist = px.histogram(SR, x='Returns', 
                        title=f'{stock_symbol} Daily Returns Distribution',
                        nbins=50,
                        labels={'Returns': 'Daily Returns'},
                        marginal='box')  # adds a box plot on top
fig_hist.update_xaxes(title_text='Daily Returns')
fig_hist.update_yaxes(title_text='Frequency')
fig_hist.show()


### Why are daily returns centered around zero even though the stock grew?

**Daily returns measure day-to-day percentage changes**, not cumulative growth. Even though AAPL has grown significantly from 2020 to 2025, on any given day:
- Sometimes the stock goes up (positive return)
- Sometimes it goes down (negative return)
- These daily fluctuations average out to be close to zero

The **long-term growth** comes from the fact that the positive days slightly outweigh the negative days, or the positive returns are slightly larger than the negative ones. Over approximately 1,450 trading days (5 years), even a small average daily return compounds to substantial growth.

For example:
- A daily return of 0.1% means $(1.001)^{252} \approx 1.29$ or 29% annual growth
- But 0.1% is very close to zero on a histogram scale that ranges from -10% to +10%

**Key insight**: Small positive mean daily returns + compounding over time = large cumulative returns!


### Calculating Excess Returns

**Excess returns** represent the return of an investment above the risk-free rate. They answer the question: *"How much extra return am I getting for taking on risk?"*

The risk-free rate is typically represented by:
- 10-year Treasury bonds (long-term investments)
- 3-month Treasury bills (short-term investments)

**Formula**: 
$$\text{Excess Return} = \text{Actual Return} - \text{Risk-Free Rate}$$

In the CAPM model, we use excess returns because:
1. **Risk compensation**: Investors should only be rewarded for taking systematic (market) risk, not for the time value of money (captured by the risk-free rate)
2. **Better comparison**: Allows us to compare investments on a level playing field
3. **Beta estimation**: The relationship between stock and market excess returns gives us beta

We'll calculate excess returns for both the stock and the market index by subtracting the risk-free rate from their daily returns.


In [65]:
# Calculate excess returns
# Convert annual risk-free rate to daily rate
daily_risk_free_rate = risk_free_rate / 252  # 252 trading days per year

print(f"Annual risk-free rate: {risk_free_rate:.2%}")
print(f"Daily risk-free rate: {daily_risk_free_rate:.6f} ({daily_risk_free_rate:.4%})")

excess_stock_returns = stock_returns - daily_risk_free_rate
excess_market_returns = market_returns - daily_risk_free_rate
excess_market_returns


Annual risk-free rate: 2.00%
Daily risk-free rate: 0.000079 (0.0079%)


Date
2020-01-03   -0.007139
2020-01-06    0.003454
2020-01-07   -0.002883
2020-01-08    0.004823
2020-01-09    0.006576
                ...   
2025-10-24    0.007823
2025-10-27    0.012211
2025-10-28    0.002209
2025-10-29   -0.000123
2025-10-30   -0.009984
Name: (Close, ^GSPC), Length: 1465, dtype: float64

In [66]:
XR = pd.DataFrame({'Date': excess_market_returns.index, 'Returns': excess_market_returns})

# Create a line plot using Plotly Express
fig = px.line(XR, x='Date', y='Returns', title=f'{stock_symbol} Excess Returns')
fig.update_xaxes(title_text='Date')
fig.update_yaxes(title_text='Returns')

# Show the plot
fig.show()

In [67]:
# Summary statistics for excess returns
print("=" * 60)
print("EXCESS RETURNS SUMMARY STATISTICS")
print("=" * 60)
print(f"\n{stock_symbol} Excess Returns:")
print(f"  Mean:   {excess_stock_returns.mean():.6f} ({excess_stock_returns.mean()*252:.2%} annualized)")
print(f"  Median: {excess_stock_returns.median():.6f}")
print(f"  Std:    {excess_stock_returns.std():.6f} ({excess_stock_returns.std()*np.sqrt(252):.2%} annualized)")
print(f"  Min:    {excess_stock_returns.min():.6f}")
print(f"  Max:    {excess_stock_returns.max():.6f}")

print(f"\n{market_index_symbol} Excess Returns:")
print(f"  Mean:   {excess_market_returns.mean():.6f} ({excess_market_returns.mean()*252:.2%} annualized)")
print(f"  Median: {excess_market_returns.median():.6f}")
print(f"  Std:    {excess_market_returns.std():.6f} ({excess_market_returns.std()*np.sqrt(252):.2%} annualized)")
print(f"  Min:    {excess_market_returns.min():.6f}")
print(f"  Max:    {excess_market_returns.max():.6f}")
print("=" * 60)


EXCESS RETURNS SUMMARY STATISTICS

AAPL Excess Returns:
  Mean:   0.001026 (25.87% annualized)
  Median: 0.001093
  Std:    0.020274 (32.18% annualized)
  Min:    -0.128727
  Max:    0.153209

^GSPC Excess Returns:
  Mean:   0.000514 (12.96% annualized)
  Median: 0.000837
  Std:    0.013310 (21.13% annualized)
  Min:    -0.119920
  Max:    0.095075


In [76]:
# Create a comparison histogram of excess returns
excess_returns_df = pd.DataFrame({
    'Date': excess_stock_returns.index,
    stock_symbol: excess_stock_returns.values,
    market_index_symbol: excess_market_returns.values
})

# Reshape for plotly
excess_returns_long = excess_returns_df.melt(id_vars='Date', 
                                              value_vars=[stock_symbol, market_index_symbol],
                                              var_name='Asset', 
                                              value_name='Excess Returns')

fig_excess_hist = px.histogram(excess_returns_long, 
                                x='Excess Returns', 
                                color='Asset',
                                title='Distribution of Excess Returns: Stock vs Market',
                                nbins=50,
                                barmode='overlay',
                                opacity=0.7)
fig_excess_hist.update_xaxes(title_text='Excess Returns')
fig_excess_hist.update_yaxes(title_text='Frequency')
fig_excess_hist.add_vline(x=0, line_dash="dash", line_color="black", annotation_text="Zero", annotation_position="top left")
fig_excess_hist.show()


### Estimating Beta Using Linear Regression

**What is Beta?**

Beta (β) measures how much a stock's returns move in relation to the market's returns. It's a measure of **systematic risk** - the risk that cannot be diversified away.

**How do we calculate Beta?**

We use **Ordinary Least Squares (OLS) regression** with:
- **Dependent variable (Y)**: Stock's excess returns
- **Independent variable (X)**: Market's excess returns

The regression equation is:
$$R_{stock} - R_f = \alpha + \beta \times (R_{market} - R_f) + \epsilon$$

Where:
- $\beta$ is the **slope coefficient** - this is our beta!
- $\alpha$ is the **intercept** (Jensen's alpha - measures abnormal returns)
- $\epsilon$ is the error term

**The CAPM Formula**

Once we have beta, we can calculate the expected return using CAPM:
$$E(R_i) = R_f + \beta_i \times (E(R_m) - R_f)$$

This tells us: *Given the stock's risk (β), what return should we expect?*

The formula says:
1. Start with the risk-free rate (what you'd get with zero risk)
2. Add a risk premium: β times the market's excess return
3. High β → High expected return (to compensate for higher risk)


In [69]:
# Perform linear regression to estimate beta
X = sm.add_constant(excess_market_returns)
model = sm.OLS(excess_stock_returns, X).fit()
# Get beta from the slope coefficient (second parameter)
alpha = model.params.iloc[0]
beta = model.params.iloc[1]


In [70]:
# Calculate expected return using CAPM
market_return = np.mean(excess_market_returns)
expected_return = risk_free_rate + beta * (market_return - risk_free_rate)

In [71]:
print(f"Stock Alpha: {alpha:.6f}")
print(f"Stock Beta: {beta:.3f}")
print(f"Expected Return: {expected_return:.3f}")

Stock Alpha: 0.000411
Stock Beta: 1.197
Expected Return: -0.003


### Understanding Alpha, Beta, and Expected Return

**Alpha (Jensen's Alpha)** is a measure of a stock's performance relative to what is predicted by the CAPM model. It represents the portion of return that cannot be explained by market movements or systematic risk. A positive alpha means the stock has outperformed its expected return (after adjusting for risk), while a negative alpha means it has underperformed. In practice, alpha is often interpreted as the value added (or lost) by active management, unique company factors, or other influences beyond broad market trends.


**What is Alpha (Jensen's Alpha)?**

Alpha (α) = **0.000411** represents the stock's **abnormal return** - the excess return beyond what CAPM predicts:
- α > 0: Stock outperformed what CAPM expected (good stock picking!)
- α = 0: Stock performed exactly as CAPM predicted
- α < 0: Stock underperformed CAPM expectations

Our alpha of 0.000411 per day equals **0.104% annualized** (0.000411 × 252), suggesting AAPL slightly outperformed what its beta would predict. This is a positive sign, though the effect is small.

**What is Beta?**

Beta (β) = **1.197** means that AAPL is **more volatile** than the market:
- β = 1.0 means the stock moves exactly with the market
- β > 1.0 means the stock is **more volatile** (amplifies market movements)
- β < 1.0 means the stock is **less volatile** (dampens market movements)

With β = 1.197, when the market goes up 1%, AAPL tends to go up about 1.2%. Similarly, when the market drops 1%, AAPL tends to drop about 1.2%.

**What is Expected Return?**

The CAPM formula gives us:
$$E(R) = R_f + \beta \times (R_m - R_f)$$

Where:
- $R_f$ = Risk-free rate (2% annual)
- $R_m$ = Market return (from our data)
- $\beta$ = Stock's beta (1.197)

**Why is Expected Return negative (-0.003)?**

This seems counterintuitive, but remember:
1. This is the **expected daily return** based on CAPM, not actual historical return
2. The market's excess return during this period might have been negative or very small
3. CAPM uses the risk-free rate and beta to predict what the return *should be*, not what it *was*
4. The negative value suggests the market didn't perform well above the risk-free rate during this period

**Key Insight**: AAPL has high systematic risk (β = 1.197), making it sensitive to market swings. The small positive alpha suggests AAPL has delivered slightly better returns than its beta would predict, possibly due to company-specific factors like innovation, strong financials, or market dominance.


### AAPL's Weight in the S&P 500

An interesting question: How much of the S&P 500 is AAPL? 

Since the S&P 500 is a **market-capitalization weighted** index, larger companies have more influence on the index's performance. AAPL, being one of the largest companies in the world, represents a significant portion of the S&P 500.

We can estimate AAPL's weight by comparing its market capitalization to the total market cap of the S&P 500 over time.


In [72]:
# Get market cap data for AAPL
# We'll use shares outstanding and price to estimate market cap over time
aapl_info = yf.Ticker(stock_symbol)
spy_info = yf.Ticker('SPY')  # S&P 500 ETF as proxy

# Get current info
print("Current AAPL Information:")
print(f"  Market Cap: ${aapl_info.info.get('marketCap', 'N/A'):,}")
print(f"  Shares Outstanding: {aapl_info.info.get('sharesOutstanding', 'N/A'):,}")

# Calculate approximate market cap over time using price * shares outstanding
# Note: shares outstanding changes over time due to buybacks/splits, 
# so this is an approximation
shares_outstanding = aapl_info.info.get('sharesOutstanding', 15000000000)  # approximate

# Calculate AAPL market cap over time
aapl_market_cap = stock_data[('Close', stock_symbol)] * shares_outstanding

# For S&P 500 total market cap, we can use SPY ETF as a proxy
# SPY tracks ~$500B with approximately 500 stocks
# A rough estimate: multiply SPY price by a scaling factor
# Better approach: use actual index divisor if available
spy_data = yf.download('SPY', start=start_date, end=end_date)


Current AAPL Information:


/var/folders/wx/mgl11c114vv7vpz1f_b7szwr0000gn/T/ipykernel_37133/3529371485.py:23: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed

  Market Cap: $4,053,006,024,704
  Shares Outstanding: 14,776,353,000


In [73]:
# Estimate AAPL's weight in S&P 500 over time
# Using a simplified approach: AAPL market cap / (SPY price * scaling factor)
# The S&P 500 index divisor is proprietary, so we'll estimate

# Get SPY close prices
spy_close = spy_data[('Close', 'SPY')] if isinstance(spy_data.columns, pd.MultiIndex) else spy_data['Close']

# Approximate S&P 500 total market cap
# SPY tracks about 1/10th of each S&P 500 stock, so rough scaling
# Better estimate: use ~40 trillion as approximate S&P 500 market cap
# Scale based on index level changes
sp500_base_marketcap = 40e12  # $40 trillion approximate
sp500_base_price = market_data[('Close', market_index_symbol)].iloc[-1]
sp500_market_cap = (market_data[('Close', market_index_symbol)] / sp500_base_price) * sp500_base_marketcap

# Calculate AAPL weight
aapl_weight = (aapl_market_cap / sp500_market_cap) * 100

# Create dataframe for plotting
weight_df = pd.DataFrame({
    'Date': aapl_weight.index,
    'AAPL Weight (%)': aapl_weight.values
})

# Plot AAPL's weight over time
fig_weight = px.line(weight_df, x='Date', y='AAPL Weight (%)',
                     title=f'{stock_symbol} Weight in S&P 500 Over Time')
fig_weight.update_xaxes(title_text='Date')
fig_weight.update_yaxes(title_text='Weight in S&P 500 (%)')
fig_weight.add_hline(y=aapl_weight.mean(), line_dash="dash", 
                     annotation_text=f"Average: {aapl_weight.mean():.2f}%",
                     line_color="red")
fig_weight.show()

print(f"\nAPPL Weight in S&P 500:")
print(f"  Start ({weight_df['Date'].iloc[0].strftime('%Y-%m-%d')}): {aapl_weight.iloc[0]:.2f}%")
print(f"  End ({weight_df['Date'].iloc[-1].strftime('%Y-%m-%d')}): {aapl_weight.iloc[-1]:.2f}%")
print(f"  Average: {aapl_weight.mean():.2f}%")
print(f"  Maximum: {aapl_weight.max():.2f}% on {aapl_weight.idxmax().strftime('%Y-%m-%d')}")
print(f"  Change: {aapl_weight.iloc[-1] - aapl_weight.iloc[0]:.2f} percentage points")



APPL Weight in S&P 500:
  Start (2020-01-02): 5.61%
  End (2025-10-30): 10.02%
  Average: 8.89%
  Maximum: 10.85% on 2023-06-30
  Change: 4.41 percentage points


### Comparing with NVIDIA (NVDA)

Let's compare AAPL's weight with another tech giant: NVIDIA. NVDA has seen explosive growth in recent years, especially with the AI boom. How does its weight in the S&P 500 compare to AAPL?


In [74]:
# Download NVIDIA data
nvda_symbol = 'NVDA'
nvda_data = yf.download(nvda_symbol, start=start_date, end=end_date)
nvda_info = yf.Ticker(nvda_symbol)

# Get NVDA info
print("Current NVDA Information:")
print(f"  Market Cap: ${nvda_info.info.get('marketCap', 'N/A'):,}")
print(f"  Shares Outstanding: {nvda_info.info.get('sharesOutstanding', 'N/A'):,}")

# Calculate NVDA market cap over time
nvda_shares_outstanding = nvda_info.info.get('sharesOutstanding', 25000000000)  # approximate
nvda_market_cap = nvda_data[('Close', nvda_symbol)] * nvda_shares_outstanding

# Calculate NVDA weight in S&P 500
nvda_weight = (nvda_market_cap / sp500_market_cap) * 100

print(f"\nNVDA Weight in S&P 500:")
print(f"  Start ({nvda_weight.index[0].strftime('%Y-%m-%d')}): {nvda_weight.iloc[0]:.2f}%")
print(f"  End ({nvda_weight.index[-1].strftime('%Y-%m-%d')}): {nvda_weight.iloc[-1]:.2f}%")
print(f"  Average: {nvda_weight.mean():.2f}%")
print(f"  Maximum: {nvda_weight.max():.2f}% on {nvda_weight.idxmax().strftime('%Y-%m-%d')}")
print(f"  Change: {nvda_weight.iloc[-1] - nvda_weight.iloc[0]:.2f} percentage points")


/var/folders/wx/mgl11c114vv7vpz1f_b7szwr0000gn/T/ipykernel_37133/2711356213.py:3: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed

Current NVDA Information:


  Market Cap: $4,705,179,664,384
  Shares Outstanding: 24,347,000,000

NVDA Weight in S&P 500:
  Start (2020-01-02): 0.76%
  End (2025-10-30): 12.35%
  Average: 4.27%
  Maximum: 12.48% on 2025-10-29
  Change: 11.59 percentage points


In [75]:
# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Date': aapl_weight.index,
    'AAPL': aapl_weight.values,
    'NVDA': nvda_weight.values
})

# Reshape for plotting
comparison_long = comparison_df.melt(id_vars='Date', 
                                      value_vars=['AAPL', 'NVDA'],
                                      var_name='Company', 
                                      value_name='Weight (%)')

# Create comparison line plot
fig_comparison = px.line(comparison_long, x='Date', y='Weight (%)', 
                         color='Company',
                         title='AAPL vs NVDA: Weight in S&P 500 Over Time')
fig_comparison.update_xaxes(title_text='Date')
fig_comparison.update_yaxes(title_text='Weight in S&P 500 (%)')
fig_comparison.show()

# Summary comparison
print("\n" + "="*60)
print("COMPARISON: AAPL vs NVDA in S&P 500")
print("="*60)
print(f"\nChange over period:")
print(f"  AAPL: {aapl_weight.iloc[0]:.2f}% → {aapl_weight.iloc[-1]:.2f}% ({aapl_weight.iloc[-1] - aapl_weight.iloc[0]:+.2f} pp)")
print(f"  NVDA: {nvda_weight.iloc[0]:.2f}% → {nvda_weight.iloc[-1]:.2f}% ({nvda_weight.iloc[-1] - nvda_weight.iloc[0]:+.2f} pp)")
print(f"\nCurrent weights:")
print(f"  AAPL: {aapl_weight.iloc[-1]:.2f}%")
print(f"  NVDA: {nvda_weight.iloc[-1]:.2f}%")
print(f"  Combined: {aapl_weight.iloc[-1] + nvda_weight.iloc[-1]:.2f}%")
print("="*60)



COMPARISON: AAPL vs NVDA in S&P 500

Change over period:
  AAPL: 5.61% → 10.02% (+4.41 pp)
  NVDA: 0.76% → 12.35% (+11.59 pp)

Current weights:
  AAPL: 10.02%
  NVDA: 12.35%
  Combined: 22.37%
